# 02 · Data frames

The pandas `DataFrame` is Python's `data.frame` / tibble. This chapter covers getting data in, looking at it, and the three ideas that differ most from R: the **Series**, the **index**, and how **missing values** behave. The dplyr-style verbs come in chapter 03.

1. Reading data
2. Inspecting
3. Columns and Series
4. Rows and the index
5. Types
6. Missing values

In [ ]:
import numpy as np
import pandas as pd

## 1. Reading data

**In R**

```r
library(readr)

penguins <- read_csv("../data/penguins.csv")
penguins
```

**In Python**

In [ ]:
penguins = pd.read_csv("../data/penguins.csv")
penguins

Like a tibble, a large data frame prints only its first and last few rows, with the dimensions underneath.

| R | Python |
| --- | --- |
| `read_csv("f.csv")` | `pd.read_csv("f.csv")` |
| `read_tsv("f.tsv")` | `pd.read_csv("f.tsv", sep="\t")` |
| `readxl::read_excel("f.xlsx")` | `pd.read_excel("f.xlsx")` (needs `uv add openpyxl`) |
| `write_csv(df, "f.csv")` | `df.to_csv("f.csv", index=False)` |

The `index=False` is explained in section 4.

### Building a data frame by hand

**In R**

```r
tibble(
  name = c("Ada", "Grace", "Linus"),
  age = c(36, 45, 28)
)
```

**In Python**

Pass a dict whose keys are the column names and whose values are lists.

In [ ]:
people = pd.DataFrame({
    "name": ["Ada", "Grace", "Linus"],
    "age": [36, 45, 28],
})
people

## 2. Inspecting

**In R**

```r
head(penguins)
dim(penguins)
names(penguins)
glimpse(penguins)     # or str(penguins)
summary(penguins)
```

**In Python**

In [ ]:
penguins.head()

In [ ]:
print(penguins.shape)       # (rows, columns): an attribute, no parentheses
print(len(penguins))        # number of rows
print(penguins.columns)

In [ ]:
penguins.info()           # the closest thing to glimpse() / str()

In [ ]:
penguins.describe()       # summary(), for the numeric columns

| R | Python |
| --- | --- |
| `head(df)`, `tail(df)` | `df.head()`, `df.tail()` |
| `dim(df)` | `df.shape` |
| `nrow(df)`, `ncol(df)` | `len(df)`, `df.shape[1]` |
| `names(df)` | `df.columns` |
| `glimpse(df)`, `str(df)` | `df.info()` |
| `summary(df)` | `df.describe()` |
| `View(df)` | click the data frame in Positron's Variables pane |

## 3. Columns and Series

**In R**

```r
penguins$body_mass_g
mean(penguins$body_mass_g, na.rm = TRUE)
penguins$body_mass_g / 1000
```

**In Python**

In [ ]:
mass = penguins["body_mass_g"]
mass

A single column is a **Series**. It is the NumPy array from chapter 01 with two additions, both visible in the output above: row labels down the left side (the *index*) and a name.

Everything from the NumPy section still applies: arithmetic is element-wise, and summaries are methods.

In [ ]:
print(type(mass))
print(mass.mean())
print(mass.max())
(mass / 1000).head()

Notice that `mass.mean()` gave a number even though the column contains missing values. pandas skips them by default, which is the opposite of R. Section 6 comes back to this.

Common Series methods:

| R | Python |
| --- | --- |
| `mean(x)`, `median(x)`, `sd(x)` | `s.mean()`, `s.median()`, `s.std()` |
| `min(x)`, `max(x)`, `sum(x)` | `s.min()`, `s.max()`, `s.sum()` |
| `unique(x)` | `s.unique()` |
| `n_distinct(x)` | `s.nunique()` |
| `table(x)` / `count(df, x)` | `s.value_counts()` |

In [ ]:
penguins["species"].value_counts()

### One column or several: the brackets flip

**In R**

```r
penguins[["species"]]              # a vector
penguins["species"]                # a one-column data frame
penguins[c("species", "island")]   # a two-column data frame
```

**In Python**

To select several columns, put a *list* of names inside the brackets. That is where the double brackets come from: the outer pair does the selecting and the inner pair is a list.

In [ ]:
print(type(penguins["species"]))               # one name: a Series
print(type(penguins[["species"]]))             # a list with one name: a DataFrame
penguins[["species", "island"]].head(3)        # a list with two names: a DataFrame

This is the reverse of R, where double brackets give the vector and single brackets give the data frame.

You will also see `penguins.species` in other people's code. It works for reading a column, but it fails for names containing spaces, for names that clash with a method (`df.count`, `df.mean`), and for creating new columns. This tutorial always uses `penguins["species"]`.

### Adding a column

**In R**

```r
penguins$body_mass_kg <- penguins$body_mass_g / 1000
```

**In Python**

In [ ]:
penguins["body_mass_kg"] = penguins["body_mass_g"] / 1000
penguins.head(3)

Chapter 03 shows the `mutate()`-style alternative, `.assign()`.

## 4. Rows and the index

Every data frame and Series has an **index**: a label for each row. It is the bold column on the left of every table above. By default the labels are just `0, 1, 2, ...`.

The closest thing in R is row names, which tibbles deliberately got rid of. pandas kept them and built a lot on top of them, so they are worth understanding.

### Selecting rows: `.iloc` and `.loc`

**In R**

```r
penguins[1, ]                               # first row
penguins[1:3, ]                             # first three rows
penguins[1:3, c("species", "island")]       # rows and columns
penguins[1, "species"]                      # a single cell
```

**In Python**

There are two accessors, and both use square brackets:

- **`.iloc`** selects by **position** (*i* for integer). It follows the list rules from chapter 01: start at 0, exclude the end.
- **`.loc`** selects by **label**: index labels for rows, column names for columns.

In [ ]:
penguins.iloc[0]                 # first row, returned as a Series

In [ ]:
penguins.iloc[0:3]               # first three rows

In [ ]:
penguins.loc[0:2, ["species", "island"]]    # rows labelled 0 to 2, two columns

In [ ]:
penguins.loc[0, "species"]       # a single cell

**Watch out:** `.loc[0:2]` returned *three* rows. Slices by label include both ends, because `.loc` is matching labels, not counting positions. `.iloc[0:2]` would return two.

### When position and label disagree

With the default index, row 0 is also labelled 0, so the two accessors look interchangeable. They stop agreeing as soon as rows are filtered or reordered, because **rows keep their labels**.

In [ ]:
heaviest = penguins.sort_values("body_mass_g", ascending=False).head(3)
heaviest

The labels on the left are `169, 185, 269`: the positions these rows had in the original data.

- `heaviest.iloc[0]` is the first row of this table (the 6300 g penguin).
- `heaviest.loc[169]` is the same row, asked for by label.
- `heaviest.loc[0]` is an error, because no row here is labelled 0.

In [ ]:
print(heaviest.iloc[0]["body_mass_g"])
print(heaviest.loc[169, "body_mass_g"])

To throw the old labels away and renumber from 0, use `.reset_index(drop=True)`:

In [ ]:
heaviest.reset_index(drop=True)

### Results that come back with a meaningful index

You will meet the index most often in the *output* of summaries. Grouping puts the group names in the index:

In [ ]:
mean_mass = penguins.groupby("species")["body_mass_g"].mean()
mean_mass

This is a Series whose index is the species names, a bit like a named vector in R. You can look values up by label:

In [ ]:
print(mean_mass.loc["Gentoo"])

And `.reset_index()` (without `drop=True`) turns the index back into an ordinary column, giving the tidy data frame that `summarise()` would have returned:

In [ ]:
mean_mass.reset_index()

### Practical advice

- Leave the default index alone for everyday work.
- When a result comes back with something useful in the index, call `.reset_index()` to get a plain table.
- When writing a CSV, use `df.to_csv("f.csv", index=False)`. Otherwise the index is written as an unnamed first column, exactly like forgetting `row.names = FALSE` in base R's `write.csv()`.

| To get | R | Python |
| --- | --- | --- |
| first row | `df[1, ]` | `df.iloc[0]` |
| first three rows | `df[1:3, ]` | `df.iloc[:3]` |
| a cell by position | `df[1, 2]` | `df.iloc[0, 1]` |
| a cell by column name | `df[1, "x"]` | `df.loc[0, "x"]` (label 0) |
| rows and named columns | `df[1:3, c("x", "y")]` | `df.loc[0:2, ["x", "y"]]` |

## 5. Types

**In R**

```r
sapply(penguins, class)     # or glimpse(penguins)
```

**In Python**

In [ ]:
penguins.dtypes

| R | pandas | Notes |
| --- | --- | --- |
| `dbl` (numeric) | `float64` | |
| `int` | `int64` | |
| `chr` | `str` | Shown as `object` in pandas versions before 3.0, and in a lot of older code and tutorials. |
| `lgl` | `bool` | |
| `fct` | `category` | |
| `date`, `dttm` | `datetime64` | |

Two things in the output above are worth a second look.

**`flipper_length_mm` and `body_mass_g` are floats**, even though every value is a whole number. A standard integer column cannot hold a missing value, so pandas reads any integer column with gaps as `float64`. `year` has no gaps, so it stayed `int64`. In R, integer vectors can hold `NA`, so this never comes up.

**`species` is `str`, not `category`.** pandas never converts strings to categories on its own.

### Converting types

**In R**

```r
as.character(penguins$year)
as.factor(penguins$species)
```

**In Python**

In [ ]:
print(penguins["year"].astype(str).head(3))
print(penguins["year"].astype(float).head(3))

### Factors become categories

**In R**

```r
species <- factor(penguins$species)
levels(species)

factor(penguins$species, levels = c("Gentoo", "Chinstrap", "Adelie"))
```

**In Python**

In [ ]:
species = penguins["species"].astype("category")
print(species.cat.categories)        # levels()

ordered = pd.Categorical(
    penguins["species"],
    categories=["Gentoo", "Chinstrap", "Adelie"],
    ordered=True,
)
print(ordered.categories)

Categories matter less in Python than factors do in R. Plotting and modelling libraries treat a plain string column as categorical automatically, so most people leave strings as strings and only reach for `category` to control the *order* of groups in a plot or a sort, or to save memory on a very large column.

## 6. Missing values

R has `NA`. pandas shows missing values as `NaN` ("not a number", borrowed from NumPy) in both numeric and text columns. You may also run into `None`, `pd.NA` and `NaT` (for dates). Treat them all as "missing": the functions below handle every kind.

**In R**

```r
is.na(penguins$sex)
sum(is.na(penguins$sex))
colSums(is.na(penguins))
```

**In Python**

In [ ]:
print(penguins["sex"].isna().sum())     # True counts as 1, as in R
penguins.isna().sum()                   # missing values per column

### Dropping and filling

**In R**

```r
drop_na(penguins)
drop_na(penguins, body_mass_g)
replace_na(penguins$sex, "unknown")
```

**In Python**

In [ ]:
print(penguins.dropna().shape)                   # rows with no missing values at all
print(penguins.dropna(subset=["body_mass_g"]).shape)     # rows where body mass is present
penguins["sex"].fillna("unknown").value_counts()

None of these changed `penguins`. Like dplyr verbs, they return a new object, and you keep the result by assigning it.

### Three behaviours that differ from R

**1. Summaries skip missing values by default.** In R, `mean(x)` returns `NA` unless you add `na.rm = TRUE`. In pandas it is the other way round: missing values are ignored unless you pass `skipna=False`.

In [ ]:
print(penguins["body_mass_g"].mean())
print(penguins["body_mass_g"].mean(skipna=False))

This is convenient, but it means a column full of gaps will not announce itself the way it does in R. Check `.isna().sum()` on unfamiliar data.

**2. Comparisons with a missing value are `False`, not `NA`.** R uses three-valued logic: `NA > 4000` is `NA`. pandas has only `True` and `False`.

In [ ]:
print(np.nan > 4000)
print(np.nan == np.nan)       # a missing value is not even equal to itself

The second line is why you must never test for missing values with `==`. `penguins["sex"] == np.nan` is `False` for every row. Always use `.isna()` (or its opposite, `.notna()`).

**3. `value_counts()` leaves missing values out**, like `table()` and unlike `count()`. Ask for them with `dropna=False`:

In [ ]:
penguins["sex"].value_counts(dropna=False)

## Recap

| R | Python |
| --- | --- |
| `read_csv("f.csv")` | `pd.read_csv("f.csv")` |
| `tibble(a = c(1, 2))` | `pd.DataFrame({"a": [1, 2]})` |
| `glimpse(df)` / `summary(df)` | `df.info()` / `df.describe()` |
| `df$x` | `df["x"]` (a Series) |
| `df[c("x", "y")]` | `df[["x", "y"]]` |
| `df$z <- ...` | `df["z"] = ...` |
| `df[1:3, ]` | `df.iloc[:3]` |
| `df[1, "x"]` | `df.loc[0, "x"]` |
| row names | the index; `df.reset_index()` to turn it into a column |
| `as.character(x)` | `s.astype(str)` |
| `factor(x)` | `s.astype("category")` |
| `is.na(x)` | `s.isna()` |
| `mean(x, na.rm = TRUE)` | `s.mean()` |
| `drop_na(df)` | `df.dropna()` |
| `replace_na(x, 0)` | `s.fillna(0)` |

---
Next: work through `exercises/02_dataframes.ipynb`, then move on to `03_data_wrangling.ipynb`.